In [1]:
# Standard library imports
import os
import shutil
import glob
from pathlib import Path
import logging


# Third-party library imports
import micom
from micom import Community
from cobra import Reaction, Metabolite, Model
from cobra.io import load_json_model, save_json_model, load_matlab_model, save_matlab_model, read_sbml_model, write_sbml_model
import cobra
import pandas as pd


# Load individual models
#model_bin1 = micom.util.load_model('models/bin.1.xml')
#model_bin2 = micom.util.load_model('models/bin.2.xml')
#model_bin3 = micom.util.load_model('models/bin.3.xml')
#model_bin4 = micom.util.load_model('models/bin.4.xml')
#model_bin5 = micom.util.load_model('models/bin.5.xml')
#model_bin6 = micom.util.load_model('models/bin.6.xml')
#model_bin7 = micom.util.load_model('models/bin.7.xml')
#model_bin8 = micom.util.load_model('models/bin.8.xml') 

In [2]:


# params
model_db_path = Path("./models/")       # Replace with your actual model directory path
adjusted_models_path = Path("./adjusted_models/")  # Replace with your desired output directory path


medium = {
    'EX_cpd00063_e0': 100.0,     # Ca+
    'EX_cpd00099_e0': 100.0,     # Cl-
    'EX_cpd00137_e0': 5.0,       # Citrate
    'EX_cpd00009_e0': 100.0,     # Phosphate
    'EX_cpd00205_e0': 100.0,     # K+
    'EX_cpd11632_e0': 100.0,     # Photon
    'EX_cpd00254_e0': 100.0,     # Mg
    'EX_cpd10516_e0': 1.0,       # Fe3+
    'EX_cpd00013_e0': 100.0,     # Ammonium
    'EX_cpd00007_e0': 100.0,     # O2
    'EX_cpd00149_e0': 100.0,     # Co2+
    'EX_cpd00058_e0': 100.0,     # Cu2+
    'EX_cpd00067_e0': 1.0,       # H+
    'EX_cpd00030_e0': 100.0,     # Mn2
    'EX_cpd00034_e0': 100.0,     # Zn2
    'EX_cpd00048_e0': 100.0,     # Sulfate
}

data = {
    'reaction': [
        'EX_cpd00063_e0', 'EX_cpd00099_e0', 'EX_cpd00137_e0', 'EX_cpd00009_e0', 
        'EX_cpd00205_e0', 'EX_cpd11632_e0', 'EX_cpd00254_e0', 'EX_cpd10516_e0', 
        'EX_cpd00013_e0', 'EX_cpd00007_e0', 'EX_cpd00149_e0', 'EX_cpd00058_e0', 
        'EX_cpd00067_e0', 'EX_cpd00030_e0', 'EX_cpd00034_e0', 'EX_cpd00048_e0'
    ],
    'flux': [
        100.0, 100.0, 5.0, 100.0, 
        100.0, 100.0, 100.0, 1.0, 
        100.0, 100.0, 100.0, 100.0, 
        1.0, 100.0, 100.0, 100.0
    ],
    'metabolite': [
        'Ca+', 'Cl-', 'Citrate', 'Phosphate', 
        'K+', 'Photon', 'Mg', 'Fe3+', 
        'Ammonium', 'O2', 'Co2+', 'Cu2+', 
        'H+', 'Mn2', 'Zn2', 'Sulfate'
    ]
}

df_medium = pd.DataFrame(data)


In [3]:
# ----------------------------
# Configure Logging
# ----------------------------

# First, get the root logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)  # Set the desired logging level

# Remove any existing handlers to prevent duplicate logs
if logger.hasHandlers():
    logger.handlers.clear()

# Create a file handler to save logs to a file
file_handler = logging.FileHandler('model_processing.log')
file_handler.setLevel(logging.INFO)

# Create a stream handler to print logs to the Jupyter cell
stream_handler = logging.StreamHandler()
stream_handler.setLevel(logging.INFO)

# Define a common formatter
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

# Assign the formatter to both handlers
file_handler.setFormatter(formatter)
stream_handler.setFormatter(formatter)

# Add both handlers to the logger
logger.addHandler(file_handler)
logger.addHandler(stream_handler)



def adjust_medium(model, medium):
    """
    Adjust the medium for the given model based on available exchange reactions.
    """
    adjusted_medium = {}
    available_exchanges = {rxn.id for rxn in model.exchanges}
    for rxn_id, value in medium.items():
        if rxn_id in available_exchanges:
            adjusted_medium[rxn_id] = value
        else:
            logging.warning(f"Exchange reaction {rxn_id} not found in model {model.id}")
    return adjusted_medium

def add_exchange_reaction(model, metabolite_id_base, lower_bound=-1000, upper_bound=1000):
    """
    Adds an exchange reaction for a metabolite, creating external and transport reactions if necessary.
    """
    # Build metabolite IDs
    internal_met_id = f"M_{metabolite_id_base}_c0"
    external_met_id = f"M_{metabolite_id_base}_e0"
    exchange_rxn_id = f"EX_M_{metabolite_id_base}_e0"
    transport_rxn_id = f"TRANS_M_{metabolite_id_base}"
    
    # Check if the internal metabolite exists
    try:
        internal_met = model.metabolites.get_by_id(internal_met_id)
    except KeyError:
        logging.error(f"Internal metabolite {internal_met_id} not found in model {model.id}. Cannot proceed.")
        return
    
    # Check if the external metabolite exists; if not, create it
    external_met = model.metabolites.get_by_id(external_met_id, None)
    if external_met is None:
        external_met = Metabolite(
            id=external_met_id,
            formula=internal_met.formula,
            name=internal_met.name,
            compartment='e0'
        )
        model.add_metabolites([external_met])
        logging.info(f"Added external metabolite {external_met_id} to model {model.id}")
    
    # Add the transport reaction if it doesn't exist
    if transport_rxn_id not in model.reactions:
        transport_rxn = Reaction(id=transport_rxn_id)
        transport_rxn.name = f"Transport of {internal_met.name}"
        transport_rxn.lower_bound = -1000
        transport_rxn.upper_bound = 1000
        transport_rxn.add_metabolites({internal_met: -1, external_met: 1})
        model.add_reactions([transport_rxn])
        logging.info(f"Added transport reaction {transport_rxn_id} to model {model.id}")
    
    # Add the exchange reaction if it doesn't exist
    if exchange_rxn_id not in model.reactions:
        exchange_rxn = Reaction(id=exchange_rxn_id)
        exchange_rxn.name = f"Exchange of {external_met.name}"
        exchange_rxn.lower_bound = lower_bound
        exchange_rxn.upper_bound = upper_bound
        exchange_rxn.add_metabolites({external_met: -1})
        model.add_reactions([exchange_rxn])
        logging.info(f"Added exchange reaction {exchange_rxn_id} to model {model.id}")
    else:
        logging.info(f"Exchange reaction {exchange_rxn_id} already exists in model {model.id}")

def add_all_exchange_reactions(model, medium):
    """
    Add exchange reactions for all components defined in the medium.
    """
    for rxn_id in medium.keys():
        if rxn_id not in model.reactions:
            # Infer metabolite ID by removing 'EX_' prefix
            if rxn_id.startswith('EX_'):
                metabolite_id = rxn_id[3:]
                if metabolite_id in model.metabolites:
                    add_exchange_reaction(model, metabolite_id, reaction_id=rxn_id)
                else:
                    logging.warning(f"Metabolite {metabolite_id} corresponding to {rxn_id} not found in model {model.id}. Exchange reaction {rxn_id} not added.")
            else:
                logging.warning(f"Reaction ID {rxn_id} does not follow the 'EX_' prefix convention. Skipping.")

def process_model(model_path, output_dir, medium):
    """
    Load a model, adjust it by adding exchange reactions for all medium components,
    set the medium, optimize the model, and save the adjusted model to the output directory.
    """
    try:
        # Load the model using micom.util.load_model (keeping it as is)
        model = micom.util.load_model(str(model_path))
        logging.info(f"\nProcessing model: {model.id}")

        # Add exchange reactions for all medium components
        add_all_exchange_reactions(model, medium)

        # Adjust the medium
        adjusted_medium = adjust_medium(model, medium)
        model.medium = adjusted_medium

        # Optional: Optimize the model to ensure it can grow
        solution = model.optimize()
        logging.info(f"Model {model.id} growth rate after adjustment: {solution.objective_value}")

        # Save the adjusted model using the existing save function
        output_model_path = output_dir / model_path.name
        write_sbml_model(model, output_model_path)  # Keeping the saving function as is
        logging.info(f"Saved adjusted model to {output_model_path}")

    except Exception as e:
        logging.error(f"Failed to process model {model_path.name}: {e}")


"""
Main function to process all models in the specified directory.
"""
# Iterate through all files in the model_db_path
for model_file in model_db_path.iterdir():
    # Check if the file is an XML file (common format for metabolic models)
    if model_file.suffix.lower() in ['.xml']:
        process_model(model_file, adjusted_models_path, medium)
    else:
        logging.warning(f"Skipping unsupported file format: {model_file.name}")

logging.info("\nAll models have been processed.")

# ----------------------------
# Copy Manifest File to the Output Directory
# ----------------------------

# Define the manifest file name (adjust if your manifest file has a different name)
manifest_filename = "manifest.csv"  # Replace with the actual manifest file name if different

# Define the source and destination paths for the manifest file
source_manifest = model_db_path / manifest_filename
destination_manifest = adjusted_models_path / manifest_filename

# Check if the manifest file exists in the source directory
if source_manifest.exists():
    try:
        shutil.copy2(source_manifest, destination_manifest)
        logging.info(f"Copied manifest file to {destination_manifest}")
    except Exception as e:
        logging.error(f"Failed to copy manifest file: {e}")
else:
    logging.warning(f"Manifest file {manifest_filename} not found in {model_db_path}. No manifest copied.")

2025-10-02 16:48:38,982 - INFO - reading model from models/bin.2.xml
2025-10-02 16:48:42,184 - INFO - 
Processing model: bin_2
2025-10-02 16:48:42,195 - INFO - Compartment `e0` sounds like an external compartment. Using this one without counting boundary reactions.
2025-10-02 16:48:42,209 - INFO - Compartment `e0` sounds like an external compartment. Using this one without counting boundary reactions.
2025-10-02 16:48:42,307 - INFO - Model bin_2 growth rate after adjustment: 0.313354579139377
2025-10-02 16:48:43,957 - INFO - Saved adjusted model to adjusted_models/bin.2.xml
2025-10-02 16:48:44,038 - INFO - reading model from models/bin.3.xml
2025-10-02 16:48:48,251 - INFO - 
Processing model: bin_3
2025-10-02 16:48:48,260 - INFO - Compartment `e0` sounds like an external compartment. Using this one without counting boundary reactions.
2025-10-02 16:48:48,280 - INFO - Compartment `e0` sounds like an external compartment. Using this one without counting boundary reactions.
2025-10-02 16:

In [4]:
manifest = pd.read_csv(model_db_path / "manifest.csv")
manifest.head(10)


,id,file,domain,phylum,class,order,family,genus,species,summary_rank
0,bin.1,bin.1.xml,Bacteria,Proteobacteria,Alphaproteobacteria,Caulobacterales,Caulobacteraceae,Caulobacter,NaN,genus
1,bin.2,bin.2.xml,Bacteria,Proteobacteria,Gammaproteobacteria,Burkholderiales,Burkholderiaceae,Limnobacter,sp002954425,genus
2,bin.3,bin.3.xml,Bacteria,Proteobacteria,Gammaproteobacteria,Burkholderiales,Burkholderiaceae,Hydrogenophaga,NaN,genus
3,bin.4,bin.4.xml,Bacteria,Bacteroidota,Bacteroidia,Chitinophagales,Chitinophagaceae,Phnomibacter,NaN,genus
4,bin.5,bin.5.xml,Bacteria,Bacteroidota,Bacteroidia,Chitinophagales,Chitinophagaceae,Sediminibacterium,NaN,genus
5,bin.6,bin.6.xml,Bacteria,Bacteroidota,Bacteroidia,NS11-12g,UBA8524,SB11,sp008933805,genus
6,bin.7,bin.7.xml,Bacteria,Cyanobacteria,Cyanobacteriia,Cyanobacteriales,Aphanizomenonaceae,Aphanizomenon,flos-aquae,genus
7,bin.8,bin.8.xml,Bacteria,Proteobacteria,Alphaproteobacteria,Sphingomonadales,Sphingomonadaceae,Sphingorhabdus_B,NaN,genus


In [5]:
from micom.workflows import build
taxonomy = pd.DataFrame({
    'id': ['bin.1', 'bin.2', 'bin.3', 'bin.4','bin.5', 'bin.6', 'bin.7', 'bin.8'],
    'sample_id': ['sample_1', 'sample_1', 'sample_1', 'sample_1','sample_1', 'sample_1', 'sample_1', 'sample_1'],
    'abundance': [0.125, 0.125,0.125,0.125,0.125,0.125,0.125,0.125],
    'phylum': ['Proteobacteria', 'Proteobacteria', 'Proteobacteria', 'Bacteroidota', 'Bacteroidota', 'Bacteroidota', 'Cyanobacteria', 'Proteobacteria'],
    'file': [
    'adjusted_models/bin.1.xml', 
    'adjusted_models/bin.2.xml', 
    'adjusted_models/bin.3.xml', 
    'adjusted_models/bin.4.xml', 
    'adjusted_models/bin.5.xml', 
    'adjusted_models/bin.6.xml', 
    'adjusted_models/bin.7.xml', 
    'adjusted_models/bin.8.xml'
]})

# Create the community model
community = Community(taxonomy)

2025-10-02 16:49:15,693 - INFO - building new micom model None.
2025-10-02 16:49:15,694 - INFO - using the cplex solver.
2025-10-02 16:49:15,696 - INFO - 0 individuals with abundances below threshold


Output()

2025-10-02 16:49:15,776 - INFO - reading model from adjusted_models/bin.1.xml
2025-10-02 16:49:19,023 - INFO - converting IDs for bin_1
2025-10-02 16:49:19,034 - INFO - Compartment `e0` sounds like an external compartment. Using this one without counting boundary reactions.
2025-10-02 16:49:19,039 - INFO - Identified e0 as the external compartment for bin_1. If that is wrong you may be in trouble...
2025-10-02 16:49:21,526 - INFO - adding reactions for bin_1 to community
2025-10-02 16:49:22,591 - INFO - adding metabolite cpd00009_m to external medium
2025-10-02 16:49:22,594 - INFO - adding metabolite cpd00023_m to external medium
2025-10-02 16:49:22,596 - INFO - adding metabolite cpd00027_m to external medium
2025-10-02 16:49:22,599 - INFO - adding metabolite cpd00028_m to external medium
2025-10-02 16:49:22,601 - INFO - adding metabolite cpd00034_m to external medium
2025-10-02 16:49:22,604 - INFO - adding metabolite cpd00039_m to external medium
2025-10-02 16:49:22,606 - INFO - addin

In [6]:
manifest = build(taxonomy, model_db=None, out_folder="community_models") 
manifest.head(10)

[10/02/25 16:50:08] WARNING  Found existing models for 1 samples. Will skip those. Delete the output    ]8;id=639953;file:///mnt/workspace/miniconda3/envs/micom/lib/python3.9/site-packages/micom/workflows/build.py\build.py]8;;\:]8;id=397964;file:///mnt/workspace/miniconda3/envs/micom/lib/python3.9/site-packages/micom/workflows/build.py#98\98]8;;\
                             folder if you would like me to rebuild them.                                          

2025-10-02 16:50:08,955 - WARNING - Found existing models for 1 samples. Will skip those. Delete the output folder if you would like me to rebuild them.


Output()

,sample_id,abundance,file
0,sample_1,0.125,sample_1.pickle


In [7]:

#rename medium compounds for comunity
def transform_medium(medium_dict):
    """
    Transforms the keys of the medium dictionary from 'EX_cpdXXXXX_e0' to 'cpdXXXXX_m'.

    Parameters:
    - medium_dict (dict): Original medium dictionary with keys like 'EX_cpdXXXXX_e0'.

    Returns:
    - dict: New dictionary with keys transformed to 'cpdXXXXX_m'.
    """
    transformed_dict = {}
    for key, value in medium_dict.items():
        if key.startswith('EX_') and key.endswith('_e0'):
            compound_id = key[3:-3]  # Extract 'cpdXXXXX' from 'EX_cpdXXXXX_e0'
            new_key = f"EX_{compound_id}_m"  # Form the new key 'cpdXXXXX_m'
            transformed_dict[new_key] = value
        else:
            # Handle keys that don't match the expected format
            transformed_dict[key] = value
    return transformed_dict
medium4com = transform_medium(medium)
medium_df = pd.DataFrame(transform_medium(medium).items(), columns=['reaction', 'flux'])


In [8]:
medium_df

,reaction,flux
0,EX_cpd00063_m,100.0
1,EX_cpd00099_m,100.0
2,EX_cpd00137_m,5.0
3,EX_cpd00009_m,100.0
4,EX_cpd00205_m,100.0
5,EX_cpd11632_m,100.0
6,EX_cpd00254_m,100.0
7,EX_cpd10516_m,1.0
8,EX_cpd00013_m,100.0
9,EX_cpd00007_m,100.0


In [9]:
from micom.workflows import tradeoff

tradeoff_rates = tradeoff(manifest, model_folder="community_models", medium=medium_df, threads=2)
tradeoff_rates.head()

Output()

,abundance,growth_rate,reactions,metabolites,taxon,tradeoff,sample_id
compartments,,,,,,,
bin_1,0.125,0.000000,1867,1781,bin_1,NaN,sample_1
bin_2,0.125,0.000000,1836,1712,bin_2,NaN,sample_1
bin_3,0.125,0.000000,2353,2046,bin_3,NaN,sample_1
bin_4,0.125,1.343744,1840,1588,bin_4,NaN,sample_1
bin_5,0.125,0.620673,1623,1432,bin_5,NaN,sample_1


In [10]:
tradeoff_rates.groupby("tradeoff").apply(
    lambda df: (df.growth_rate > 1e-6).sum()).reset_index()

/tmp/ipykernel_891476/957046585.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tradeoff_rates.groupby("tradeoff").apply(


,tradeoff,0
0,0.1,8
1,0.2,8
2,0.3,8
3,0.4,8
4,0.5,8
5,0.6,8
6,0.7,8
7,0.8,8
8,0.9,8
9,1.0,8


In [11]:
from micom.viz import plot_tradeoff

pl = plot_tradeoff(tradeoff_rates, filename="tradeoff.html")

2025-10-02 16:50:27,106 - INFO - Writing visualization to tradeoff.html.


In [12]:
from micom.workflows import grow
res = grow(manifest, model_folder="community_models", medium=medium_df, tradeoff=0.9, threads=2)

Output()

In [13]:
from micom.viz import plot_growth

pl = plot_growth(res, filename="growth_rates.html")
pl.view()

2025-10-02 16:50:35,272 - INFO - Writing visualization to growth_rates.html.


>4;1H84l=                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [14]:
from micom.viz import plot_focal_interactions

pl = plot_focal_interactions(res, taxon="bin_7", filename='bin_7_interactions.html')



2025-10-02 16:50:35,934 - INFO - Writing visualization to bin_7_interactions.html.


In [15]:
from micom.interaction import MES

scores = MES(res)
scores.head()



,metabolite,sample_id,MES,name,molecular_weight,C_number,N_number,sbo,metanetx.chemical,inchikey,seed.compound,hmdb,reactome,kegg.compound,chebi,bigg.metabolite,biocyc,reaction
0,cpd00001_e0,sample_1,3.750000,H2O-e0,18.015280,0,0,SBO:0000247,MNXM2,XLYOFNOQVPJJNP-UHFFFAOYSA-N,cpd00001,"['HMDB01039', 'HMDB02111']","['R-ALL-109276', 'R-ALL-113518', 'R-ALL-113519...","['C00001', 'C01328']",CHEBI:15377,h2o,META:WATER,EX_cpd00001_e0
1,cpd00003_e0,sample_1,2.857143,NAD-e0,662.417162,21,7,SBO:0000247,MNXM8,BAWFJGJZGIEFAR-NNYOXOHSSA-M,cpd00003,HMDB00902,"['R-ALL-113526', 'R-ALL-192307', 'R-ALL-194653...",C00003,CHEBI:57540,nad,META:NAD,EX_cpd00003_e0
2,cpd00006_e0,sample_1,2.666667,NADP-e0,740.381183,21,7,SBO:0000247,MNXM5,XJLXINKUBYWONI-NNYOXOHSSA-K,cpd00006,HMDB00217,"['R-ALL-113563', 'R-ALL-113564', 'R-ALL-194668...",C00006,CHEBI:58349,nadp,META:NADP,EX_cpd00006_e0
3,cpd00007_e0,sample_1,3.000000,O2-e0,31.998800,0,0,SBO:0000247,MNXM4,MYMOFIZGZYHOMD-UHFFFAOYSA-N,cpd00007,HMDB01377,"['R-ALL-1131511', 'R-ALL-113533', 'R-ALL-11353...",C00007,CHEBI:15379,o2,META:OXYGEN-MOLECULE,EX_cpd00007_e0
4,cpd00009_e0,sample_1,0.000000,Phosphate-e0,95.979301,0,0,SBO:0000247,MNXM9,NBIIXXVUZAFLBC-UHFFFAOYSA-L,cpd00009,"['HMDB00973', 'HMDB01429', 'HMDB05947', 'HMDB0...","['R-ALL-109277', 'R-ALL-113548', 'R-ALL-113550...",C00009,CHEBI:43474,pi,"['META:CPD-16459', 'META:PHOSPHATE-GROUP', 'ME...",EX_cpd00009_e0


In [16]:

from micom.viz import plot_mes


pl = plot_mes(res, filename="mes.html")



2025-10-02 16:50:36,021 - INFO - Writing visualization to mes.html.


In [17]:
exch_df = res.exchanges
exch_df

,taxon,sample_id,tolerance,reaction,flux,abundance,metabolite,direction
0,bin_1,sample_1,0.000001,EX_cpd00130_e0,22.293177,0.125,cpd00130_e0,export
1,bin_1,sample_1,0.000001,EX_cpd00024_e0,-14.732060,0.125,cpd00024_e0,import
2,bin_1,sample_1,0.000001,EX_cpd00107_e0,-0.087801,0.125,cpd00107_e0,import
4,bin_1,sample_1,0.000001,EX_cpd00007_e0,-6.778556,0.125,cpd00007_e0,import
5,bin_1,sample_1,0.000001,EX_cpd00100_e0,4.140715,0.125,cpd00100_e0,export
...,...,...,...,...,...,...,...,...
1309,medium,sample_1,0.000001,EX_cpd00205_m,-0.003402,NaN,cpd00205_m,import
1313,medium,sample_1,0.000001,EX_cpd00013_m,-7.823702,NaN,cpd00013_m,import
1336,medium,sample_1,0.000001,EX_cpd00058_m,-0.003402,NaN,cpd00058_m,import
1345,medium,sample_1,0.000001,EX_cpd00034_m,-0.003402,NaN,cpd00034_m,import


In [18]:
print(res.exchanges)
bin_7_exchanges = (res.exchanges[res.exchanges.taxon == "bin_7"])

       taxon sample_id  tolerance        reaction       flux  abundance  \
0      bin_1  sample_1   0.000001  EX_cpd00130_e0  22.293177      0.125   
1      bin_1  sample_1   0.000001  EX_cpd00024_e0 -14.732060      0.125   
2      bin_1  sample_1   0.000001  EX_cpd00107_e0  -0.087801      0.125   
4      bin_1  sample_1   0.000001  EX_cpd00007_e0  -6.778556      0.125   
5      bin_1  sample_1   0.000001  EX_cpd00100_e0   4.140715      0.125   
...      ...       ...        ...             ...        ...        ...   
1309  medium  sample_1   0.000001   EX_cpd00205_m  -0.003402        NaN   
1313  medium  sample_1   0.000001   EX_cpd00013_m  -7.823702        NaN   
1336  medium  sample_1   0.000001   EX_cpd00058_m  -0.003402        NaN   
1345  medium  sample_1   0.000001   EX_cpd00034_m  -0.003402        NaN   
1367  medium  sample_1   0.000001   EX_cpd00001_m  13.163844        NaN   

       metabolite direction  
0     cpd00130_e0    export  
1     cpd00024_e0    import  
2     cpd

In [19]:

def map_metabolite_names(df, reaction_col='reaction', new_col='metabolite_name', mapping_data=None):
    """
    Maps metabolite names to reactions in any DataFrame using a separate mapping data dictionary.

    Parameters:
        df (pd.DataFrame): The DataFrame to process.
        reaction_col (str): Name of the column in df that contains reaction codes.
        new_col (str): Name for the new column that will contain mapped metabolite names.
        mapping_data (dict): A dictionary with keys 'reaction' and 'metabolite' to build the mapping.

    Returns:
        pd.DataFrame: df with an added column for metabolite names.
    """
    if mapping_data is None or not all(k in mapping_data for k in ('reaction', 'metabolite')):
        raise ValueError("mapping_data must be a dictionary with 'reaction' and 'metabolite' keys.")

    # Build the lookup dictionary
    reaction_to_metabolite = dict(zip(mapping_data['reaction'], mapping_data['metabolite']))

    # Map and create new column
    df[new_col] = df[reaction_col].map(reaction_to_metabolite)

    return df



In [20]:
annotated_df = map_metabolite_names((res.exchanges[res.exchanges.taxon == "bin_7"]), mapping_data=data)
annotated_df

/tmp/ipykernel_891476/2953825138.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[new_col] = df[reaction_col].map(reaction_to_metabolite)


,taxon,sample_id,tolerance,reaction,flux,abundance,metabolite,direction,metabolite_name
926,bin_7,sample_1,0.000001,EX_cpd00130_e0,-3.107530,0.125,cpd00130_e0,import,NaN
932,bin_7,sample_1,0.000001,EX_cpd00007_e0,19.998152,0.125,cpd00007_e0,export,O2
933,bin_7,sample_1,0.000001,EX_cpd00039_e0,-0.151921,0.125,cpd00039_e0,import,NaN
934,bin_7,sample_1,0.000001,EX_cpd00220_e0,-0.003684,0.125,cpd00220_e0,import,NaN
935,bin_7,sample_1,0.000001,EX_cpd00051_e0,-0.130949,0.125,cpd00051_e0,import,NaN
939,bin_7,sample_1,0.000001,EX_cpd00036_e0,-10.417058,0.125,cpd00036_e0,import,NaN
941,bin_7,sample_1,0.000001,EX_cpd00363_e0,15.986510,0.125,cpd00363_e0,export,NaN
942,bin_7,sample_1,0.000001,EX_cpd00324_e0,-0.073556,0.125,cpd00324_e0,import,NaN
944,bin_7,sample_1,0.000001,EX_cpd00224_e0,-0.565858,0.125,cpd00224_e0,import,NaN
946,bin_7,sample_1,0.000001,EX_cpd00025_e0,0.001848,0.125,cpd00025_e0,export,NaN


In [21]:
from micom.interaction import interactions

results = res
ints = interactions(results, taxa="bin_7")
ints.sort_values(by="flux", ascending=False).head()
from micom.interaction import summarize_interactions

summary = summarize_interactions(ints)
summary.head()

,sample_id,focal,partner,class,flux,mass_flux,C_flux,N_flux,n_ints
0,sample_1,bin_7,bin_1,co-consumed,4.909865,0.311760,10.770050,1.629363,28
1,sample_1,bin_7,bin_1,provided,4.741860,0.400084,16.872036,1.296470,9
2,sample_1,bin_7,bin_1,received,3.502834,0.503556,20.040289,0.075808,12
3,sample_1,bin_7,bin_2,co-consumed,4.864982,1.362564,87.432810,1.454175,29
4,sample_1,bin_7,bin_2,provided,6.002731,0.297473,8.579646,1.288408,10


In [22]:
from micom.interaction import interactions, summarize_interactions
full = interactions(results, taxa=None, threads=8)
full.shape

full_summary = summarize_interactions(full)
full_summary.shape


Output()

(168, 9)

In [23]:
full_summary

,sample_id,focal,partner,class,flux,mass_flux,C_flux,N_flux,n_ints
0,sample_1,bin_1,bin_2,co-consumed,18.804334,1.857076,56.453670,7.961352,36
1,sample_1,bin_1,bin_2,provided,2.354991,0.326258,14.460314,0.119348,10
2,sample_1,bin_1,bin_2,received,0.908516,0.157815,10.100177,0.086414,4
3,sample_1,bin_1,bin_3,co-consumed,4.809585,0.379668,15.521913,2.259347,33
4,sample_1,bin_1,bin_3,provided,11.802236,1.415454,47.137034,5.440692,13
...,...,...,...,...,...,...,...,...,...
163,sample_1,bin_8,bin_6,provided,7.523174,0.947967,32.138254,7.997216,12
164,sample_1,bin_8,bin_6,received,11.802669,0.880384,29.173876,6.917745,12
165,sample_1,bin_8,bin_7,co-consumed,6.323878,0.405353,13.168440,1.633052,22
166,sample_1,bin_8,bin_7,provided,3.362381,0.334655,14.934031,0.141187,12


In [25]:


from micom import Community, data
from micom.elasticity import elasticities

tax = pd.DataFrame({
    'id': ['bin.1', 'bin.2', 'bin.3', 'bin.4','bin.5', 'bin.6', 'bin.7', 'bin.8'],
    'sample_id': ['sample_1', 'sample_1', 'sample_1', 'sample_1','sample_1', 'sample_1', 'sample_1', 'sample_1'],
    'abundance': [0.125, 0.125,0.125,0.125,0.125,0.125,0.125,0.125],
    'phylum': ['Proteobacteria', 'Proteobacteria', 'Proteobacteria', 'Bacteroidota', 'Bacteroidota', 'Bacteroidota', 'Cyanobacteria', 'Proteobacteria'],
    'file': [
    'adjusted_models/bin.1.xml', 
    'adjusted_models/bin.2.xml', 
    'adjusted_models/bin.3.xml', 
    'adjusted_models/bin.4.xml', 
    'adjusted_models/bin.5.xml', 
    'adjusted_models/bin.6.xml', 
    'adjusted_models/bin.7.xml', 
    'adjusted_models/bin.8.xml'
]})
com = Community(tax)

eps = elasticities(com, fraction=1.0, reactions=com.exchanges)
eps.head()



2025-10-02 16:51:49,840 - INFO - building new micom model None.
2025-10-02 16:51:49,842 - INFO - using the cplex solver.
2025-10-02 16:51:49,844 - INFO - 0 individuals with abundances below threshold


Output()

2025-10-02 16:51:49,856 - INFO - reading model from adjusted_models/bin.1.xml
2025-10-02 16:51:53,024 - INFO - converting IDs for bin_1
2025-10-02 16:51:53,034 - INFO - Compartment `e0` sounds like an external compartment. Using this one without counting boundary reactions.
2025-10-02 16:51:53,036 - INFO - Identified e0 as the external compartment for bin_1. If that is wrong you may be in trouble...
2025-10-02 16:51:55,205 - INFO - adding reactions for bin_1 to community
2025-10-02 16:51:55,951 - INFO - adding metabolite cpd00009_m to external medium
2025-10-02 16:51:55,955 - INFO - adding metabolite cpd00023_m to external medium
2025-10-02 16:51:55,957 - INFO - adding metabolite cpd00027_m to external medium
2025-10-02 16:51:55,959 - INFO - adding metabolite cpd00028_m to external medium
2025-10-02 16:51:55,962 - INFO - adding metabolite cpd00034_m to external medium
2025-10-02 16:51:55,965 - INFO - adding metabolite cpd00039_m to external medium
2025-10-02 16:51:55,967 - INFO - addin

2025-10-02 16:52:42,586 - INFO - adding L2 norm to None
2025-10-02 16:52:42,594 - INFO - finished adding tradeoff objective to None
2025-10-02 16:52:43,376 - INFO - solver returned the status numeric, returning the solution anyway.
2025-10-02 16:52:43,439 - INFO - Starting crossover...
2025-10-02 16:52:43,441 - INFO - constraining growth rates.
2025-10-02 16:52:43,450 - INFO - finding closest feasible solution


Output()

2025-10-02 16:52:44,907 - INFO - solver returned the status numeric, returning the solution anyway.
2025-10-02 16:52:44,977 - INFO - Starting crossover...
2025-10-02 16:52:44,978 - INFO - constraining growth rates.
2025-10-02 16:52:44,989 - INFO - finding closest feasible solution
2025-10-02 16:52:46,352 - INFO - solver returned the status numeric, returning the solution anyway.
2025-10-02 16:52:46,419 - INFO - Starting crossover...
2025-10-02 16:52:46,421 - INFO - constraining growth rates.
2025-10-02 16:52:46,430 - INFO - finding closest feasible solution
2025-10-02 16:52:47,802 - INFO - solver returned the status numeric, returning the solution anyway.
2025-10-02 16:52:47,863 - INFO - Starting crossover...
2025-10-02 16:52:47,864 - INFO - constraining growth rates.
2025-10-02 16:52:47,874 - INFO - finding closest feasible solution
2025-10-02 16:52:49,227 - INFO - solver returned the status numeric, returning the solution anyway.
2025-10-02 16:52:49,296 - INFO - Starting crossover...

2025-10-02 16:53:05,771 - INFO - adding L2 norm to None
2025-10-02 16:53:05,785 - INFO - finished adding tradeoff objective to None
2025-10-02 16:53:06,565 - INFO - solver returned the status numeric, returning the solution anyway.
2025-10-02 16:53:06,626 - INFO - Starting crossover...
2025-10-02 16:53:06,627 - INFO - constraining growth rates.
2025-10-02 16:53:06,636 - INFO - finding closest feasible solution


Output()

2025-10-02 16:53:07,170 - INFO - setting new abundances for None
2025-10-02 16:53:07,172 - INFO - updating exchange reactions for None
2025-10-02 16:53:07,355 - INFO - updating the community objective for None
2025-10-02 16:53:08,218 - INFO - solver returned the status numeric, returning the solution anyway.
2025-10-02 16:53:08,284 - INFO - Starting crossover...
2025-10-02 16:53:08,285 - INFO - constraining growth rates.
2025-10-02 16:53:08,296 - INFO - finding closest feasible solution
2025-10-02 16:53:08,857 - INFO - setting new abundances for None
2025-10-02 16:53:08,859 - INFO - updating exchange reactions for None
2025-10-02 16:53:09,051 - INFO - updating the community objective for None
2025-10-02 16:53:09,062 - INFO - setting new abundances for None
2025-10-02 16:53:09,064 - INFO - updating exchange reactions for None
2025-10-02 16:53:09,263 - INFO - updating the community objective for None
2025-10-02 16:53:10,081 - INFO - solver returned the status numeric, returning the solut

,reaction,taxon,effector,direction,elasticity,type
0,EX_cpd00009_m,medium,EX_cpd00009_m,reverse,0.0,exchanges
1,EX_cpd00023_m,medium,EX_cpd00009_m,zero,0.0,exchanges
2,EX_cpd00027_m,medium,EX_cpd00009_m,zero,0.0,exchanges
3,EX_cpd00028_m,medium,EX_cpd00009_m,zero,0.0,exchanges
4,EX_cpd00034_m,medium,EX_cpd00009_m,reverse,0.0,exchanges


In [36]:
# Filter rows where elasticity is not zero and effector is EX_cpd11606_m
filtered_eps = eps[(eps['elasticity'] != 0)]

with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(filtered_eps)


           reaction   taxon       effector direction    elasticity       type
828   EX_cpd00009_m  medium  EX_cpd10516_m   reverse  3.122747e-01  exchanges
829   EX_cpd00023_m  medium  EX_cpd10516_m      zero  1.558075e-05  exchanges
830   EX_cpd00027_m  medium  EX_cpd10516_m      zero  1.917307e-06  exchanges
831   EX_cpd00028_m  medium  EX_cpd10516_m      zero  6.993304e-06  exchanges
832   EX_cpd00034_m  medium  EX_cpd10516_m   reverse -2.220066e-03  exchanges
833   EX_cpd00039_m  medium  EX_cpd10516_m      zero  1.115655e-05  exchanges
834   EX_cpd00041_m  medium  EX_cpd10516_m      zero  2.289181e-05  exchanges
835   EX_cpd00048_m  medium  EX_cpd10516_m   reverse -3.523353e-03  exchanges
836   EX_cpd00051_m  medium  EX_cpd10516_m      zero  1.202138e-05  exchanges
837   EX_cpd00053_m  medium  EX_cpd10516_m      zero  1.558822e-05  exchanges
838   EX_cpd00064_m  medium  EX_cpd10516_m      zero  1.266837e-05  exchanges
839   EX_cpd00118_m  medium  EX_cpd10516_m      zero  1.260665e-